# Week 7 Problem Set: The Vendor's Headline Number

**Instructions:** Complete all four tasks.

You are auditing a digital ad vendor's effectiveness report. Your job: reproduce the vendor's headline, compute the valid numbers, and write a recommendation to the client.

---

**Lying with data: the checklist so far**

1. **W1:** Conflating fixed and marginal costs to make a tactic look cheaper than it is.
2. **W2:** Presenting an observational comparison as a causal effect.
3. **W3:** Applying a result from one setting to a different one without argument.
4. **W4:** Cherry-picking the winning arm from a multi-arm test.
5. **W5:** Treating an underpowered null as evidence of no effect.
6. **W7:** Reporting the complier comparison as if it were a causal effect.

### Before you start

**Save your own copy first.** Go to **File → Save a copy in Drive**. A new tab opens with your own copy. Work in that tab; edits to the original are not saved.

**The data loads itself.** There is nothing to download or upload. The setup cell below pulls the data straight from the course repository; you just need to be online.

Stuck? See the Colab troubleshooting guide on the syllabus.

## Setup

In [ ]:
import pandas as pd
import numpy as np

In [ ]:
df = pd.read_csv('https://raw.githubusercontent.com/joshuakalla/data_science_campaigns_26/'
                 'main/weeks/wk07_itt_vs_late/data/ad_experiment.csv')
df.shape

## Quick review

Your Week 5 problem set was due at the start of today’s class, so start with a warm-up. In the cell below, filter `df` to just the treatment group (`assigned_treatment == 1`) and compute the mean of `turned_out`. One line is enough.

*Check: the treatment group turnout should be around 45%.*

In [ ]:
# Quick review: one line
treatment_turnout = # YOUR CODE HERE
treatment_turnout

## Task 1: Reproduce the vendor's headline

The code below computes the vendor’s number: turnout among voters who viewed the ad vs. turnout among the control group. **Run it.**

In [ ]:
# The vendor's comparison: viewers vs. control
viewer_turnout = df[(df['assigned_treatment'] == 1) &
                    (df['viewed_ad'] == 1)]['turned_out'].mean()
control_turnout = df[df['assigned_treatment'] == 0]['turned_out'].mean()
naive_gap = viewer_turnout - control_turnout

print(f'Viewer turnout:   {100*viewer_turnout:.1f}%')
print(f'Control turnout:  {100*control_turnout:.1f}%')
print(f'Vendor headline:  {100*naive_gap:+.1f} pp')

**Question 1a:** In one sentence, what is the vendor comparing?

*Your answer:*

## Task 1b: Visualize the selection bias

Make a bar chart showing turnout for three groups: ad viewers, non-viewers (treatment group members who didn’t see the ad), and the control group. This is your first chart on a problem set.

The code below is pre-filled. **Run it.** Then write one sentence: what does the chart tell you about who the viewers are?

In [ ]:
import matplotlib.pyplot as plt

# Turnout for three groups
viewer_to = df[(df['assigned_treatment'] == 1) & (df['viewed_ad'] == 1)]['turned_out'].mean()
nonviewer_to = df[(df['assigned_treatment'] == 1) &
                  (df['viewed_ad'] == 0)]['turned_out'].mean()
control_to = df[df['assigned_treatment'] == 0]['turned_out'].mean()

groups = ['Ad viewers\n(treatment)', 'Non-viewers\n(treatment)', 'Control']
turnouts = [100 * viewer_to, 100 * nonviewer_to, 100 * control_to]
colors = ['#e74c3c', '#95a5a6', '#3498db']

fig, ax = plt.subplots(figsize=(7, 4))
ax.bar(groups, turnouts, color=colors, edgecolor='white', width=0.6)
ax.set_ylabel('Turnout (%)')
ax.set_title('Turnout by group: the vendor compared red to blue')
ax.set_ylim(0, 60)
for j, v in enumerate(turnouts):
    ax.text(j, v + 1, f'{v:.1f}%', ha='center', fontsize=11)
plt.tight_layout()
plt.show()

**Your one sentence:** what does this chart reveal about the vendor’s comparison?

*Your answer:*

**Before you move on:** The code above filters to `(df['assigned_treatment'] == 1) & (df['viewed_ad'] == 1)` to get the viewers. What kind of voter is most likely to be in this group? Would you expect these voters to have higher or lower turnout than average, *even if the ad did nothing*? In 1–2 sentences, explain why this makes the vendor’s comparison unreliable.

**Your answer:**

*Replace this text with your answer.*

## Task 2: Compute the ITT

The intent-to-treat effect compares everyone *assigned* to treatment vs. everyone assigned to control, regardless of whether they viewed the ad.

In the empty cell below, compute the ITT using a `groupby` on `assigned_treatment`. This is the same pattern you’ve used since Week 3. **Store the answer in a variable called `itt`** — Task 3 needs it.

*Check: the ITT should be very small, around +0.1 percentage points.*

In [ ]:
itt = # YOUR CODE HERE
itt

**Question 2:** The ITT is about +0.1 pp. The vendor reported +5.5 pp. In one sentence, explain why the ITT is so much smaller.

*Your answer:*

## Task 2b: How sure are we?

Task 2 gave you a point estimate. It does not tell you how far that estimate could have bounced by chance.

The cell below is pre-filled. **Run it.** It fits the same comparison as a regression and prints four numbers: the ITT, its standard error, its p-value, and its 95% confidence interval.

*Check: the interval runs from about \N{MINUS SIGN}0.51 to +0.72 percentage points.*

In [ ]:
import statsmodels.formula.api as smf

m = smf.ols('turned_out ~ assigned_treatment', data=df).fit()
coef = 100 * m.params['assigned_treatment']
se = 100 * m.bse['assigned_treatment']
pval = m.pvalues['assigned_treatment']
lo, hi = 100 * m.conf_int().loc['assigned_treatment']

print(f'ITT      {coef:+.2f} pp')
print(f'std err   {se:.2f} pp')
print(f'p-value   {pval:.2f}')
print(f'95% CI   [{lo:+.2f}, {hi:+.2f}] pp')

**Question 2b:** The interval includes zero. In one or two sentences: what does that do to the claim that the ad worked? And does it mean the ad did nothing?

*Your answer:*

## Task 3: Compute the LATE

The LATE estimates the effect on viewers without the selection, using the formula:

**LATE = ITT / compliance rate**

In the two cells below:

1. Compute the compliance rate: the fraction of the treatment group that viewed the ad. Store it as `compliance_rate`.
2. Divide the ITT by the compliance rate to get the LATE. Store it as `late` and print all three numbers.

*Check: the compliance rate should be about 33%. The LATE should be around +0.3 pp.*

In [ ]:
# Step 1: compute the compliance rate
compliance_rate = # YOUR CODE HERE

In [ ]:
# Step 2: compute the LATE = ITT / compliance rate
late = # YOUR CODE HERE

print(itt, compliance_rate, late)

**Before you move on:** What would happen to the LATE if the compliance rate were 100% instead of 33% (every voter in the treatment group actually viewed the ad)? Would the LATE be larger, smaller, or equal to the ITT? Why?

**Your answer:**

*Replace this text with your answer.*

## Task 4: Memo to the client (250–350 words)

Your client spent \$3 million on this digital ad campaign targeting \~50,000 voters. The vendor claims a 5.5-point turnout lift. You’ve now computed the valid numbers.

Write a memo addressed to the client. Your memo must address all four of the following:

**(a) Why the vendor’s headline is wrong.** Explain what the vendor compared and why that comparison is biased. Use the specific numbers from Task 1 (viewer turnout, non-viewer turnout, control turnout) to make your case. Imagine a client who is smart but has never taken this class.

**(b) What the valid numbers are.** Report the ITT and the LATE. Explain what each one means in plain English. Compute the cost per additional vote under the ITT, and say what the confidence interval from Task 2b does to your confidence in it.

**(c) Steelman the vendor.** Construct the strongest version of the vendor’s defense: “The complier effect IS what matters. You should evaluate the ad based on what it did to people who actually saw it, not people who used ad blockers.” Then explain why this argument doesn’t rescue the +5.5 number.

**(d) What the client should do with the next \$3M.** A memo that only says the vendor is wrong does not answer the question your client asked. Pick one of the three options from the case and defend it. If your answer is “renew with conditions,” say what the condition is, in a sentence a vendor could act on.

**Important:** The vendor’s +5.5 pp is not a valid causal estimate. Do not use it as evidence that the ad works. Use the ITT or LATE instead.

**Style rules:**
- State your recommendation in the first sentence.
- 250–350 words.

**Memo to:** Client (IE Group Director)
**From:** You, Consultant
**Re:** AdReach Digital effectiveness report audit

*Replace this text with your 250–350 word memo.*

---

**Due at 4:00pm on Wednesday Oct 28, after the October recess**, to the **problem set** assignment on Canvas. Whatever you had at 5:55pm in class already went to the separate **in-class** assignment; that one is your attendance credit and you do not resubmit it.


## Before you submit

1. **Runtime → Restart session and run all.** Do this *after* you have finished every task and written your memo. It clears every variable and runs the notebook from top to bottom, in order, so the version you hand in is one that actually works start to finish.
2. **Check that every cell actually ran.** Scroll from the top. Every code cell should show a number in its left margin and its output below it. If the run stopped at a cell with an error, that is a cell you have not finished. Fix it, then restart and run all again.
3. **File → Print → Save as PDF.**
4. **Open the PDF and read it before you upload.** The PDF will look complete even when it isn't. Every heading and prompt prints whether or not the code ran. What matters is the **output**: under each code cell you should see a table, a number, or a plot. A red error box, or `In [ ]` with nothing beneath it, means that part did not run and will be graded as missing. Also check that your memo printed in full and that no plot is cut off at a page break.
5. Upload the PDF to Canvas.